# 01 — Data Exploration

This notebook performs the initial exploration of the scientific document dataset used in the project.

The project focuses on **Building a Knowledge Graph for Scientific Document Exploration using NLP and Semantic Retrieval**. Therefore, this notebook verifies that the selected arXiv metadata subset is suitable for downstream NLP tasks such as information extraction, semantic retrieval, and knowledge graph construction.

## Goals

- Load the project configuration and dataset.
- Inspect the arXiv metadata schema.
- Analyze category distribution.
- Analyze title, abstract, and combined document length.
- Save an exploration sample for later experiments.

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.common import (
    display_basic_frame_info,
    load_project_documents,
    save_table,
    setup_notebook,
)

CONFIG, PATHS = setup_notebook()

## Load dataset

In [ ]:
df, documents, dataset_path = load_project_documents(CONFIG, PATHS)

print("Dataset file:", dataset_path)
print("Loaded documents:", len(documents))

display_basic_frame_info(df, "arXiv subset")

## Validate expected schema

In [ ]:
expected_columns = {"doc_id", "title", "abstract", "authors", "categories", "date"}
missing_columns = expected_columns.difference(df.columns)

if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

print("Schema validation passed.")

## Inspect dataset configuration

In [ ]:
dataset_cfg = CONFIG.get("dataset", {})

print("Configured sample size:", dataset_cfg.get("sample_size"))
print("Configured categories:", dataset_cfg.get("categories"))
print("Random seed:", CONFIG.get("project", {}).get("seed", 42))

## Category distribution

In [ ]:
from collections import Counter

category_counter = Counter()

for categories in df["categories"]:
    category_counter.update(categories)

category_df = (
    pd.DataFrame(category_counter.items(), columns=["category", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

display(category_df.head(25))

## Visualize top categories

In [ ]:
if category_df.empty:
    print("No categories available.")
else:
    top_categories = category_df.head(15).sort_values("count")

    plt.figure(figsize=(9, 5))
    plt.barh(top_categories["category"], top_categories["count"])
    plt.title("Top arXiv Categories in the Loaded Subset")
    plt.xlabel("Number of Documents")
    plt.ylabel("Category")
    plt.tight_layout()
    plt.show()

## Text length analysis

In [ ]:
df = df.copy()
df["title_word_count"] = df["title"].fillna("").astype(str).str.split().str.len()
df["abstract_word_count"] = df["abstract"].fillna("").astype(str).str.split().str.len()
df["combined_text"] = df["title"].fillna("").astype(str) + ". " + df["abstract"].fillna("").astype(str)
df["combined_word_count"] = df["combined_text"].str.split().str.len()

df[["title_word_count", "abstract_word_count", "combined_word_count"]].describe()

## Visualize document length distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["combined_word_count"], bins=40)
plt.title("Document Length Distribution")
plt.xlabel("Words in Title + Abstract")
plt.ylabel("Number of Documents")
plt.tight_layout()
plt.show()

## Preview representative documents

In [ ]:
preview_columns = ["doc_id", "title", "categories", "date", "abstract"]
display(df[preview_columns].head(5))

## Save exploration sample

In [ ]:
output_path = PATHS.data_processed / "exploration_sample.csv"
save_table(df, output_path)

print("Saved exploration sample to:", output_path)